In [122]:
# Importy
import os, json, pickle, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pretty_midi
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
#from tqdm.notebook import tqdm 

from functions import DoomDataset

In [ ]:
# Parametry
VOCAB_SIZE = 579
EMBEDDING_DIM = 256
HIDDEN_SIZE = 256
NUM_LAYERS = 2
STRIDE = 16 
CONTEXT = 256
BATCH_SIZE = 128
EPOCHS = 35
LR = 0.001
MAX_TRANSPOSE = 12
STEPS_PER_BEAT = 12
# Zmienic przed odpaleniem treningu. 
# Wszyscy wiemy, że zajmująca maks 5 minut implementacja prostego sprawdzania, czy plik istnieje jest zbędna i niepotrzebnie komplikuje kod prawda?
FILE_NAME_SAVE = 'lstm_music_model_35ep_2Layers_256hidden_lr0001_context256_ed256_stride16_bs128__05d1_04d2_V22.pth'
FILE_NAME_LOAD = 'lstm_music_model_35ep_2Layers_256hidden_lr0001_context256_ed256_stride16_bs128__07d1_06d2_V22.pth'

#lstm_music_model_50ep_2Layers_512hidden_lr0005_context512_ed512_stride256_V7.pth
#  lstm_music_model_60ep_2Layers_512hidden_lr0005_context512_ed512_stride256_V9
# best V13/14/16_J
os.makedirs('../models', exist_ok=True)
os.makedirs('../output', exist_ok=True)
os.makedirs('../output/LSTM', exist_ok=True)

PROC_DIR = '../data/processed/'
OUTPUT_DIR = '../output/LSTM/'
MODEL_DIR = '../models/'

In [124]:
# Ustwaianie ziarna i cuda
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Device: NVIDIA GeForce RTX 4060 Laptop GPU


In [125]:
# Ładowanie danych i słownika tokenów
with open(os.path.join(PROC_DIR, 'sequences.pkl'), 'rb') as f:
    data = pickle.load(f)

with open(os.path.join(PROC_DIR, 'vocab.json')) as f:
    token2id = json.load(f)
id2token = {y: x for x,y in token2id.items()}

sequences = data.values()
names = list(data.keys())
VOCAB_SIZE = len(token2id)
PAD_ID = token2id['PAD']
print(f'Sequences: {len(sequences)} | Vocab: {VOCAB_SIZE} | Tokens: {sum(len(s) for s in sequences):,}')

Sequences: 76 | Vocab: 579 | Tokens: 931,588


In [126]:
# Przygotowanie DataLoaderów
sequences_list = list(sequences)
random.shuffle(sequences_list)
train_sequences, val_sequences = sequences_list[:int(0.8*len(sequences_list))], sequences_list[int(0.8*len(sequences_list)):]
train_dataset = DoomDataset(train_sequences, context = CONTEXT, stride = STRIDE, augment=True, max_transpose=MAX_TRANSPOSE)
print(f"Total training samples generated: {len(train_dataset)}")
print(f"With batch_size={BATCH_SIZE}, one epoch will take {len(train_dataset) // BATCH_SIZE} steps.")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = DoomDataset(val_sequences, context = CONTEXT, stride=STRIDE, augment=False, max_transpose=MAX_TRANSPOSE)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Total validation samples generated: {len(val_dataset)}")

Total training samples generated: 41331
With batch_size=128, one epoch will take 322 steps.
Total validation samples generated: 15712


In [127]:
# Pobranie jednej paczki danych
features, labels = next(iter(train_loader))

# Sprawdzenie kształtu danych
print(f"Kształt cech (X): {features.shape}")
print(f"Kształt etykiet (y): {labels.shape}")


Kształt cech (X): torch.Size([128, 256])
Kształt etykiet (y): torch.Size([128, 256])


In [ ]:
# Definicja modelu
class MusicLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super(MusicLSTM, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True, dropout=0.5)
        
        self.ln = nn.LayerNorm(hidden_size)

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden=None):
        
        embedded = self.embedding(x)
        
        out, hidden = self.lstm(embedded, hidden)
        
        out = self.ln(out)

        out = self.dropout(out)

        logits = self.fc(out)
        
        return logits, hidden
  

In [ ]:
# Inicjalizacja modelu, optymalizatora i funkcji straty
model = MusicLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS).to(device)

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1) # Poza testami zostawić na 0.1
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.08)


In [ ]:
# Trenowanie modelu
import torch
from torch.optim.lr_scheduler import StepLR

scheduler = StepLR(optimizer, step_size=12, gamma=0.5)

for epoch in range(EPOCHS):
   
    model.train()
    total_loss = 0
    total_batches = len(train_loader)

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()

        logits, _ = model(inputs)

        loss = loss_fn(logits.view(-1, VOCAB_SIZE), targets.view(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 40 == 0 or batch_idx == total_batches - 1:
            current_avg_loss = total_loss / (batch_idx + 1)
            print(f"Epoch: [{epoch+1}/{EPOCHS}] | Batch: [{batch_idx + 1}/{total_batches}] | Train Loss: {current_avg_loss:.4f}")

    epoch_train_loss = total_loss / total_batches
    print(f"Epoch {epoch+1} Train Summary: Avg Loss = {epoch_train_loss:.4f} ")

    model.eval() 
    total_val_loss = 0
    val_batches = len(val_loader)

    with torch.no_grad():
        for val_inputs, val_targets in val_loader:
            val_inputs = val_inputs.to(device)
            val_targets = val_targets.to(device)

            val_logits, _ = model(val_inputs)
            val_loss = loss_fn(val_logits.view(-1, VOCAB_SIZE), val_targets.view(-1))
            
            total_val_loss += val_loss.item()

    epoch_val_loss = total_val_loss / val_batches
    print(f"Epoch {epoch+1} Val Summary: Avg Loss = {epoch_val_loss:.4f}")

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning rate for next epoch: {current_lr}\n")


Epoch: [1/35] | Batch: [1/323] | Train Loss: 6.7542
Epoch: [1/35] | Batch: [41/323] | Train Loss: 5.2253
Epoch: [1/35] | Batch: [81/323] | Train Loss: 4.7231
Epoch: [1/35] | Batch: [121/323] | Train Loss: 4.4037
Epoch: [1/35] | Batch: [161/323] | Train Loss: 4.1920
Epoch: [1/35] | Batch: [201/323] | Train Loss: 4.0388
Epoch: [1/35] | Batch: [241/323] | Train Loss: 3.9207
Epoch: [1/35] | Batch: [281/323] | Train Loss: 3.8262
Epoch: [1/35] | Batch: [321/323] | Train Loss: 3.7476
Epoch: [1/35] | Batch: [323/323] | Train Loss: 3.7440
Epoch 1 Train Summary: Avg Loss = 3.7440 
Epoch 1 Val Summary: Avg Loss = 4.0088
Learning rate for next epoch: 0.001

Epoch: [2/35] | Batch: [1/323] | Train Loss: 3.1922
Epoch: [2/35] | Batch: [41/323] | Train Loss: 3.1427
Epoch: [2/35] | Batch: [81/323] | Train Loss: 3.1158
Epoch: [2/35] | Batch: [121/323] | Train Loss: 3.0901
Epoch: [2/35] | Batch: [161/323] | Train Loss: 3.0701
Epoch: [2/35] | Batch: [201/323] | Train Loss: 3.0518
Epoch: [2/35] | Batch: [24

In [131]:
# Zapis modelu
torch.save(model.state_dict(), os.path.join(MODEL_DIR, FILE_NAME_SAVE))

In [132]:
def is_not_percusion(token):
    return True if token < 259 or token > 385 else False
def is_note(token):
    return True if token > 2 and token < 131 else False

In [133]:
# Generacja muzyki z modyfikacjami
def generate_music(model, token2id, id2token, max_generated_tokens=2000, temperature=0.9):
    model.eval()

    last_detected_pattern = None
    pattern_repeat_count = 0
    N = 6

    generated = [token2id['BOS']]
    
    hidden = None
    change_point = random.randint(2000,4000)

    with torch.no_grad():
        for i in range(max_generated_tokens):

            
            input_tensor = torch.tensor([[generated[-1]]], dtype=torch.long).to(device)
            
            logits, hidden = model(input_tensor, hidden)
            
            logits = logits[0, -1, :]

            # Nakładanie kary na powtarzające się tokeny
            penalty = 1.0
            if change_point > 0:
                recent_tokens = set(generated[-100:]) 

                for token in recent_tokens:
                    if is_note(token):
                        logits[token] -= penalty

            # wykrywanie sekwencji
            #melody_only_history = [t for t in generated if is_note(t)]

            #if len(melody_only_history) >= N * 2:
                #current_pattern = melody_only_history[-N:]
                #previous_pattern = melody_only_history[-2*N:-N]
                
                #if current_pattern == previous_pattern:
                    #if current_pattern == last_detected_pattern:
                        #pattern_repeat_count += 1
                    #else:
                        #last_detected_pattern = current_pattern
                        #pattern_repeat_count = 2
                #else:
                    #pattern_repeat_count = 0
                    #last_detected_pattern = None

                #if pattern_repeat_count >= 4:
                    #forbidden_token = melody_only_history[-N]
                    #logits[forbidden_token] -= 4.0
            
            # Wyeliminowanie preferowanego instrumentu na starcie, by zwiększyć różnorodność
            # Bez tego większość piosenek zaczyna się identycznie i jest bardzo podobna przez całą resztę trwania melodii
            #if len(generated) < 20:
                #logits[417] = float('-inf')

            # Zablokowanie pętli perkusyjnej
            #recent_history = generated[-30:]
            #percussion_count = sum(1 for t in recent_history if not is_not_percusion(t))
            
            #if percussion_count > 18:  
                #for token_idx in range(logits.size(-1)):
                    #if not is_not_percusion(token_idx):
                        #logits[token_idx] -= 0.6

            # Dynamiczna zmiana temperatury w celu zwiększenia różnorodności
            if change_point <= 0:
                current_temp = temperature + 0.08  
                if change_point < -10:            
                    change_point = random.randint(2000, 3000)
            else:
                current_temp = temperature

            # część funckji bazowej
            logits = logits / current_temp
            
            # Dynamiczny sampling
            p = 0.96 if change_point <= 0 else 0.90
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[indices_to_remove] = float('-inf')

            # Ciąg dalszy normalnej generacji
            probabilities = F.softmax(logits, dim=-1)
            
            next_token_id = torch.multinomial(probabilities, num_samples=1).item()
            
            generated.append(next_token_id)

            change_point -= 1
    
            if next_token_id == token2id.get('EOS'):
                break
                
    generated_tokens = [id2token[idx] for idx in generated]
    return generated_tokens




In [134]:
# Generacja muzyki
def generate_music_clear(model, token2id, id2token, max_generated_tokens=2000, temperature=0.9):
    model.eval()

    last_detected_pattern = None
    pattern_repeat_count = 0
    N = 3

    generated = [token2id['BOS']]
    
    hidden = None
    change_point = random.randint(560,1000)

    with torch.no_grad():
        for i in range(max_generated_tokens):

            
            input_tensor = torch.tensor([[generated[-1]]], dtype=torch.long).to(device)
            
            logits, hidden = model(input_tensor, hidden)
            
            logits = logits[0, -1, :]

            # część funckji bazowej
            logits = logits / temperature

            # Ciąg dalszy normalnej generacji
            probabilities = F.softmax(logits, dim=-1)
            
            next_token_id = torch.multinomial(probabilities, num_samples=1).item()
            
            generated.append(next_token_id)

            change_point -= 1
    
            if next_token_id == token2id.get('EOS'):
                break
                
    generated_tokens = [id2token[idx] for idx in generated]
    return generated_tokens



In [135]:
# Generowanie utworów
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, FILE_NAME_LOAD)))

MAX_GENERATED_TOKENS = 13400
TEMPERATURE = 0.5465

generated_sequence = generate_music_clear(model, token2id, id2token, max_generated_tokens=MAX_GENERATED_TOKENS, temperature=TEMPERATURE)
print(f"Generated {len(generated_sequence)} tokens.")
print("Example fragment::", generated_sequence[:30])

Generated 13401 tokens.
Example fragment:: ['BOS', 'PROGRAM_30', 'NOTE_ON_44', 'PROGRAM_30', 'NOTE_ON_44', 'PROGRAM_30', 'NOTE_ON_44', 'PROGRAM_30', 'NOTE_ON_44', 'DRUM_51', 'DRUM_36', 'SHIFT_2', 'NOTE_OFF_44', 'NOTE_OFF_44', 'NOTE_OFF_44', 'NOTE_OFF_44', 'SHIFT_1', 'PROGRAM_30', 'NOTE_ON_44', 'PROGRAM_30', 'NOTE_ON_44', 'PROGRAM_30', 'NOTE_ON_44', 'PROGRAM_30', 'NOTE_ON_44', 'DRUM_51', 'SHIFT_2', 'NOTE_OFF_44', 'NOTE_OFF_44', 'SHIFT_1']


In [136]:
# Konwersja tokenów do MIDI
def events_to_midi(events, bpm=95):
    sec_per_step = (60.0 / bpm) / STEPS_PER_BEAT
    midi      = pretty_midi.PrettyMIDI()
    insts     = {}     # program -> Instrument (melodyczne)
    drum_inst = None   # jeden wspólny kanał perkusji
    active    = {}     # pitch -> [(start_step, program)]
    cur, last_prog = 0, 0

    for ev in events:
        if ev in ('BOS', 'EOS', 'PAD'):
            continue
        if ev.startswith('SHIFT_'):
            cur += int(ev.split('_')[1])
        elif ev.startswith('PROGRAM_'):
            last_prog = int(ev.split('_')[1])
        elif ev.startswith('NOTE_ON_'):
            pitch = int(ev.split('_')[-1])
            active.setdefault(pitch, []).append((cur, last_prog))
        elif ev.startswith('NOTE_OFF_'):
            pitch = int(ev.split('_')[-1])
            if active.get(pitch):
                start_step, prog = active[pitch].pop(0)
                start = start_step * sec_per_step
                end   = cur * sec_per_step
                if end > start:
                    if prog not in insts:
                        insts[prog] = pretty_midi.Instrument(program=prog)
                    insts[prog].notes.append(pretty_midi.Note(100, pitch, start, end))
        elif ev.startswith('DRUM_'):
            pitch = int(ev.split('_')[1])
            start = cur * sec_per_step
            end   = start + sec_per_step
            if drum_inst is None:
                drum_inst = pretty_midi.Instrument(program=0, is_drum=True)
            drum_inst.notes.append(pretty_midi.Note(100, pitch, start, end))

    for inst in insts.values():
        midi.instruments.append(inst)
    if drum_inst is not None:
        midi.instruments.append(drum_inst)
    return midi

In [137]:
# Zapis wygenerowanej muzyki do pliku MIDI

VERSION = "V68"

events_to_midi(generated_sequence, bpm = 50).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v1.mid"))
events_to_midi(generated_sequence, bpm = 75).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v2.mid"))
events_to_midi(generated_sequence, bpm = 95).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v3.mid"))
events_to_midi(generated_sequence, bpm = 120).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v4.mid"))
events_to_midi(generated_sequence, bpm = 140).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v5.mid"))
events_to_midi(generated_sequence, bpm = 165).write(os.path.join(OUTPUT_DIR, f"Generated_DOOM_{VERSION}_v6.mid"))